## 1. Combine raw Harvest meter files

This is the **first** notebook. It walks a folder of meter subfolders, cleans column names, and writes one combined CSV.

**You should get:** `../data/outputs/harvest_orig_YYMMDD-YYMMDD.csv` with columns `datetime`, `meter_name`, `kwh`, `3_phase_watt_total`.


### Enter input

Change this cell only
- **`data_path`** — folder that contains **one subfolder per meter**, each with CSV files (not a single CSV).
- **`output_dir`** — where the combined file is saved.
- **`dedup_check`** — print duplicate rows (does not delete them).
- **`m_list`** — print meter names and write `*_meter_list.csv` next to the output.
- **`name`** — first part of the output filename.

In [ ]:
# directories
input_dir = '../data/extracts/' # need if data_path 
output_dir = '../data/outputs/'

# define the data path to meter data folder ie: thumbdrive folder or local folder
data_path = '/Users/cassiehuber/Downloads/260508_meter_data' #input_dir + '260508_meter_data'

##########################################################################

# True if want duplication check
dedup_check = False

# True if want list of meters
m_list = True

##########################################################################

# original data name (for basename ie: 'harvest')
name = 'harvest' # for file naming

In [ ]:
import pandas as pd
import os, sys
import importlib

sys.path.append(os.path.abspath('..'))
import modules.harvest_orig as hv # import self defined module
import modules.file_naming as fn # import self defined module
import modules.harvest_kwh as hk
importlib.reload(hv)

<module 'modules.harvest_orig' from '/Users/cassiehuber/Documents/GitHub/harvest/modules/harvest_orig.py'>

### Load, combine, and save

1. `validate_base_path` — stop if `data_path` does not exist.
2. `load_meter_dfs` — read CSVs per meter (skips bad files and **prints** why).
3. `concat_meter_dfs` — one table for all meters.
4. Save using dates in the data.

Reads all meter CSVs under `data_path`, combines them, and writes
`output_dir/harvest_orig_YYMMDD-YYMMDD.csv`.


In [ ]:
# check if base path exists
if not hv.validate_base_path(data_path):
    print(f"Error: Path {data_path} does not exist")
    exit()

# var for file naming
var = 'orig'

# load list of meter data dataframes from CSV files in data_path
meters_df = hv.load_meter_dfs(data_path)

# combine list of all meter dataframes into one dataframe
combined_df = hv.concat_meter_dfs(meters_df)

# create orig filename
orig_filename = output_dir + fn.make_filename(combined_df, name, var, 'csv') #fix or remove startend

# convert dataframe to csv, this will be for processing kw
combined_df.to_csv(orig_filename, index=False)

# check if duplicate reading exist in resulting meter data
if dedup_check:
    hk.duplicate_check(combined_df)

# get list of meters (from csv)
if m_list:
    hv.meter_list(orig_filename)